# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method: Random Forest Classifier

The baseline rule uses one signal (position) with a hard threshold. A tree ensemble is the natural
next step because it can combine multiple prior-window signals (clicks, position, consistency,
AI share) and learn non-linear thresholds and interactions automatically — e.g. "weak position
matters more when clicks are also low" — without me hand-specifying that interaction the way the
rule required. Logistic Regression was the other candidate; Random Forest is preferred here
because ML-05's exploration already showed position's relationship to decline isn't perfectly
linear (bucketed, not continuous, in the feature vector), which trees handle natively.
Gradient Boosting was considered but skipped for now — more prone to overfitting on a small
eligible population (n=6,920), and Random Forest is the safer first model to interrogate.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [4]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from getpass import getpass
import os, duckdb, numpy as np, pandas as pd

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass('Paste your HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
tbl = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"

raw = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS max_d FROM {tbl}),
    decision AS (SELECT max_d - INTERVAL 30 DAY AS decision_date FROM bounds),
    prior AS (
        SELECT f.content_hash_id, ANY_VALUE(f.client_hash_id) AS client_hash_id,
            AVG(f.gsc_clicks) AS avg_daily_clicks_prior,
            AVG(f.gsc_avg_position) FILTER (WHERE f.gsc_avg_position > 0) AS avg_position_prior,
            COUNT(*) FILTER (WHERE f.gsc_clicks > 0) AS days_with_clicks_prior,
            SUM(f.sessions_ai) * 1.0 / NULLIF(SUM(f.sessions_organic + f.sessions_ai), 0) AS ai_share_prior,
            COUNT(*) AS n_days_prior
        FROM {tbl} f, decision d
        WHERE f.report_date >= d.decision_date - INTERVAL 90 DAY AND f.report_date < d.decision_date
        GROUP BY f.content_hash_id
        HAVING COUNT(*) >= 30
    ),
    future AS (
        SELECT f.content_hash_id, AVG(f.gsc_clicks) AS avg_daily_clicks_future
        FROM {tbl} f, decision d
        WHERE f.report_date >= d.decision_date AND f.report_date < d.decision_date + INTERVAL 30 DAY
        GROUP BY f.content_hash_id
    )
    SELECT p.*, fu.avg_daily_clicks_future
    FROM prior p JOIN future fu USING (content_hash_id)
""").df()

raw['avg_position_prior'] = raw['avg_position_prior'].fillna(100)
raw['no_ranking_data_prior'] = (raw['avg_position_prior'] == 100).astype(int)
raw['ai_share_prior'] = raw['ai_share_prior'].fillna(0)
raw['click_consistency_prior'] = raw['days_with_clicks_prior'] / raw['n_days_prior']
raw['is_declining_future'] = (
    (raw['avg_daily_clicks_future'] < 0.75 * raw['avg_daily_clicks_prior']) &
    (raw['avg_daily_clicks_prior'] >= 1.0)
).astype(int)
# ga4_coverage_prior intentionally excluded — ML-06 proved it's a client-level confound, not a page signal

eligible = raw.loc[raw['avg_daily_clicks_prior'] >= 1.0].copy()
feature_cols = ['avg_daily_clicks_prior', 'avg_position_prior', 'click_consistency_prior',
                 'ai_share_prior', 'no_ranking_data_prior']
X = eligible[feature_cols]
y = eligible['is_declining_future']
groups = eligible['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
eligible_test = eligible.iloc[test_idx].copy()

print(f'train: {len(X_train):,} rows, {groups.iloc[train_idx].nunique()} clients')
print(f'test:  {len(X_test):,} rows, {groups.iloc[test_idx].nunique()} clients')

Paste your HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train: 4,094 rows, 18 clients
test:  2,826 rows, 9 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
from sklearn.metrics import roc_auc_score

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, class_weight='balanced')
model.fit(X_train, y_train)

# --- Model Precision@50 on the test split ---
eligible_test['model_score'] = model.predict_proba(X_test)[:, 1]
model_ranked = eligible_test.sort_values('model_score', ascending=False)
model_p50 = model_ranked['is_declining_future'].iloc[:50].mean()

# --- Baseline Precision@50, recomputed on the SAME test split (fair comparison) ---
baseline_ranked = eligible_test.sort_values('avg_position_prior', ascending=False)
baseline_p50 = baseline_ranked['is_declining_future'].iloc[:50].mean()

test_base_rate = y_test.mean()

comparison = pd.DataFrame({
    'method': ['Baseline (position rule)', 'Random Forest'],
    'Precision@50': [baseline_p50, model_p50],
    'lift over base rate': [baseline_p50 / test_base_rate, model_p50 / test_base_rate],
})
print(f'test-set base rate: {test_base_rate:.3f}')
comparison

test-set base rate: 0.652


,method,Precision@50,lift over base rate
0,Baseline (position rule),0.64,0.981889
1,Random Forest,0.78,1.196678


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# Feature importances — what is the model actually leaning on?
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Feature importances:\n", importances)

# Where model and rule disagree in their top 50
model_top50_ids = set(model_ranked['content_hash_id'].iloc[:50])
baseline_top50_ids = set(baseline_ranked['content_hash_id'].iloc[:50])
only_model = model_top50_ids - baseline_top50_ids
only_baseline = baseline_top50_ids - model_top50_ids
print(f'\nflagged by model only: {len(only_model)}')
print(f'flagged by rule only: {len(only_baseline)}')

# Model's false positives in its own top 50
model_top50 = model_ranked.iloc[:50]
false_positives = model_top50[model_top50['is_declining_future'] == 0]
print(f'\nmodel false positives in top 50: {len(false_positives)}')
false_positives[['content_hash_id', 'model_score'] + feature_cols]

Feature importances:
 avg_position_prior         0.334163
click_consistency_prior    0.306824
avg_daily_clicks_prior     0.267922
ai_share_prior             0.091091
no_ranking_data_prior      0.000000
dtype: float64

flagged by model only: 44
flagged by rule only: 44

model false positives in top 50: 11


,content_hash_id,model_score,avg_daily_clicks_prior,avg_position_prior,click_consistency_prior,ai_share_prior,no_ranking_data_prior
275539,content_62850cc56d45a7cf,0.955,5.766667,6.071473,0.988889,0.000000,0
85932,content_3b8638bf882420e3,0.955,10.066667,2.242811,1.000000,0.000000,0
85155,content_dbbde91a0db0f0e3,0.950,14.477778,2.800646,1.000000,0.000000,0
96975,content_e9856d7d976aa034,0.945,22.300000,1.466810,1.000000,0.000000,0
129787,content_456b4254b508afe3,0.945,9.844444,3.238728,1.000000,0.000000,0
96864,content_9bc20f2a6e42afc4,0.935,9.877778,1.627859,1.000000,0.000000,0
263034,content_fe3fd3422852d721,0.930,16.277778,3.308166,1.000000,0.000000,0
156196,content_622c43641b04b12f,0.925,1.000000,26.280345,0.559322,0.000000,0
76752,content_2fe6474415293c3c,0.920,8.833333,2.350147,1.000000,0.000000,0
278657,content_25ffb2033b43393e,0.900,8.655556,3.281006,1.000000,0.000000,0


On a client-held-out test split, the Random Forest scored Precision@50 = 0.78–0.80 (base rate
0.652, ~1.2x lift) across two runs — note this notebook re-queries the live warehouse each
execution, so decision_date and the eligible population shift slightly run to run; small metric
movement (0.78 vs 0.80) reflects that, not model instability (random_state is fixed). The
position-only baseline, evaluated on this same honest split, scored ~0.64 — essentially parity
with random guessing, despite scoring 0.860 in ML-07 on its own in-sample population. That gap is
the more important finding: the baseline's apparent strength was partly an artifact of being
evaluated on the same clients it was designed against, the same failure mode ML-06 caught in
ga4_coverage_prior, here showing up in evaluation methodology rather than a feature.

The model leans most on avg_position_prior (0.33) — the one signal ML-06 independently validated
— followed closely by click_consistency_prior (0.31) and avg_daily_clicks_prior (0.27). Most
false positives (10-11 of 50) are pages with strong position, high consistency, and solid volume:
likely stable, established pages the model misreads as "at risk" because steady high traffic
resembles the declining-page pattern in this feature space. One false positive instead sits right
at the eligibility floor with weak position — a different failure mode, a borderline page rather
than a stable one. A content team should treat flags as directional, not causal, and expect
roughly 1 in 5 top-priority picks to be a false alarm.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.